In [0]:
%sql
USE e_commerce.bronze;

In [0]:
%run "../src/utils/common_functions"

In [0]:
%run "../src/utils/configuration"

In [0]:
%run "../src/utils/merge"

In [0]:
%run "../src/utils/rules"

In [0]:
from pyspark.sql.functions import col, current_timestamp, concat
import json

In [0]:
df_product_bronze = spark.read.format("delta").table("bronze.product")

In [0]:
display(df_product_bronze.limit(2))

In [0]:
df_product_renamed_bronze = df_product_bronze\
    .withColumnRenamed("product_category_name", "category_name")\
    .withColumnRenamed("product_name_lenght", "name_lenght")\
    .withColumnRenamed("product_description_lenght", "description_lenght")\
    .withColumnRenamed("product_photos_qty", "photos_qty")\
    .withColumnRenamed("product_weight_g", "weight_g")\
    .withColumnRenamed("product_length_cm", "length_cm")\
    .withColumnRenamed("product_height_cm", "height_cm")\
    .withColumnRenamed("product_width_cm", "width_cm")

In [0]:
try:
    last_record = get_last_record(spark, "e_commerce.silver.product")
    
    df_product_filter_bronze = df_product_renamed_bronze\
    .filter(f"ingestion_timestamp > '{last_record}'")
except Exception as e:
    print(e)

In [0]:
reglas = []

In [0]:
nulos(df_product_renamed_bronze, "product_id", reglas, "product")
duplicados(df_product_renamed_bronze, "product_id", reglas, "product")

In [0]:
df_reglas = spark.createDataFrame(reglas)
df_reglas = df_reglas.withColumn("validation_date", current_timestamp())

In [0]:
df_reglas.write.mode("append").format("delta").saveAsTable("bronze.log_transformation")

In [0]:
error = [regla for regla in reglas if not regla["cumple"]]

if error:
    dbutils.jobs.taskValues.set(key="estado", value="Error crítico")
    dbutils.jobs.taskValues.set(key="detalle", value=json.dumps(error))
else:
    dbutils.jobs.taskValues.set(key="estado", value="Validación exitosa")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("product_id") \
               .orderBy(col("ingestion_timestamp").desc())

df_product_silver = df_product_filter_bronze \
    .withColumn("rn", row_number().over(window)) \
    .filter("rn = 1") \
    .drop("rn")

In [0]:
try:
    merge(
        df_source = df_product_silver,
        target_table = "e_commerce.silver.product",
        merge_key = "product_id",
        transforms = {

        },
        exclude_update_cols = ["source_name"]
    )
except Exception as e:
    print(e)

In [0]:
%sql
SELECT file_date, COUNT(1) FROM silver.product GROUP BY file_date;

In [0]:
%sql
SELECT * FROM bronze.log_transformation WHERE tabla = 'product' ORDER BY validation_date DESC LIMIT 10;